# 04 - Vietnamese ASR (single video)

Vietnamese speech recognition from the original video using `faster-whisper large-v3`. This notebook is independent from OCR and keyframes.


In [ ]:
%pip install -q faster-whisper pandas pyarrow
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg)


In [ ]:
from __future__ import annotations

import gc
import json
import shutil
from pathlib import Path

import pandas as pd
from faster_whisper import WhisperModel

VIDEO_ID = "L21_V001"
PIPELINE_VERSION = "aicv3-asr-vi"
MODEL_NAME = "large-v3"
LANGUAGE = "vi"
BEAM_SIZE = 5
CHECKPOINT_EVERY = 25

video_matches = list(Path("/kaggle/input").rglob(f"{VIDEO_ID}.mp4"))
if not video_matches:
    raise FileNotFoundError(f"Cannot find {VIDEO_ID}.mp4 trong /kaggle/input")
VIDEO_PATH = video_matches[0]
OUTPUT_ROOT = Path("/kaggle/working/04-asr-output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_ROOT / "asr-checkpoint.jsonl"

print("Video:", VIDEO_PATH)


In [ ]:
# T4 and P100 support float16. Model initialization fails immediately when CUDA is unavailable.
model = WhisperModel(
    MODEL_NAME,
    device="cuda",
    compute_type="float16",
)
print("Whisper model ready:", MODEL_NAME)

segments_generator, info = model.transcribe(
    str(VIDEO_PATH),
    language=LANGUAGE,
    beam_size=BEAM_SIZE,
    vad_filter=True,
    vad_parameters={
        "min_silence_duration_ms": 500,
        "speech_pad_ms": 250,
    },
    condition_on_previous_text=True,
    word_timestamps=True,
)

print("Detected language:", info.language)
print("Language probability:", info.language_probability)
print("Duration:", info.duration)


In [ ]:
records = []
with CHECKPOINT_PATH.open("w", encoding="utf-8") as checkpoint:
    for segment_index, segment in enumerate(segments_generator):
        words = []
        if segment.words:
            words = [
                {
                    "start_time": float(word.start),
                    "end_time": float(word.end),
                    "word": word.word.strip(),
                    "probability": float(word.probability),
                }
                for word in segment.words
            ]
        record = {
            "segment_id": f"{VIDEO_ID}:asr:{segment_index:06d}",
            "video_id": VIDEO_ID,
            "segment_index": segment_index,
            "start_time": float(segment.start),
            "end_time": float(segment.end),
            "text": segment.text.strip(),
            "language": LANGUAGE,
            "avg_logprob": float(segment.avg_logprob),
            "no_speech_prob": float(segment.no_speech_prob),
            "words": words,
        }
        records.append(record)
        checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
        checkpoint.flush()
        if segment_index == 0 or (segment_index + 1) % CHECKPOINT_EVERY == 0:
            print(
                f"ASR segment {segment_index + 1}: "
                f"{segment.start:.2f}s-{segment.end:.2f}s | {segment.text.strip()}"
            )

if not records:
    raise RuntimeError("No speech segment was recognized in the video")

asr_df = pd.DataFrame(records).sort_values("start_time").reset_index(drop=True)
asr_df.to_json(
    OUTPUT_ROOT / "asr-segments.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

# Serialize words as JSON for a stable Parquet/PostgreSQL schema.
asr_parquet_df = asr_df.copy()
asr_parquet_df["words_json"] = asr_parquet_df["words"].map(
    lambda value: json.dumps(value, ensure_ascii=False)
)
asr_parquet_df = asr_parquet_df.drop(columns=["words"])
asr_parquet_df.to_parquet(OUTPUT_ROOT / "asr-segments.parquet", index=False)

full_transcript = " ".join(asr_df["text"].tolist())
(OUTPUT_ROOT / "transcript.txt").write_text(full_transcript, encoding="utf-8")

summary = {
    "stage": "asr",
    "pipeline_version": PIPELINE_VERSION,
    "video_id": VIDEO_ID,
    "model": f"faster-whisper/{MODEL_NAME}",
    "language": LANGUAGE,
    "language_probability": float(info.language_probability),
    "duration_sec": float(info.duration),
    "segments": int(len(asr_df)),
    "speech_duration_sec": float((asr_df["end_time"] - asr_df["start_time"]).sum()),
    "word_count": int(sum(len(value) for value in asr_df["words"])),
}
(OUTPUT_ROOT / "asr-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Preview the first 30 segments.
display(asr_df[["start_time", "end_time", "text", "avg_logprob", "no_speech_prob"]].head(30))

# Validate timestamps and package the output.
assert asr_df["segment_id"].is_unique
assert (asr_df["start_time"] >= 0).all()
assert (asr_df["end_time"] > asr_df["start_time"]).all()
assert asr_df["start_time"].is_monotonic_increasing
assert asr_df["end_time"].max() <= info.duration + 1.0

archive = shutil.make_archive(
    "/kaggle/working/04-asr-vietnamese-L21_V001",
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Download:", archive)

del model
gc.collect()
print("Released Whisper memory")
